# Training LightGBM Ensemble Meta-Models

This notebook trains two LightGBM gradient-boosting classifiers (one per label class) that serve as
the ensemble meta-model for the LGB annotation strategy.

**Inputs**: The dataset `danghaidang-passau/lgb_Data`, which contains per-model probability columns
from four LLM annotators (Mistral-7B, Llama3.1-8B, Gemma2-9B, Qwen2.5-14B) for samples in the
human-labeled training sets.

**Training**: For each label task (Hate=1 and Neutral=2), a binary LightGBM model is trained using
LLM probability outputs as features and human annotations as targets.

**Outputs**: Two trained models saved as `lgb_model_label_1.pkl` and `lgb_model_label_2.pkl`.
These are used by `Ensemble_Labeling.ipynb` to label the large-scale OWS synthetic corpus.

## 1. Imports and LightGBM Configuration

Import required libraries. The LightGBM `params` dict defines the training configuration:
- `objective: binary` — binary cross-entropy loss
- `metric: binary_logloss` — validation metric
- `boosting_type: gbdt` — gradient boosted decision tree

These settings are used for both the Hate and Neutral label classifiers.

In [2]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

import torch
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
from huggingface_hub import hf_hub_download
import lightgbm as lgb

params = {
    'objective': 'binary',  
    'metric': 'binary_logloss', 
    'boosting_type': 'gbdt',  
    'num_leaves': 34, 
    'learning_rate': 0.05,  
    'feature_fraction': 0.9, 
    'bagging_fraction': 0.8, 
    'bagging_freq': 5,  
    'verbose': -1, 
}

/tmp/ipykernel_49729/4054558623.py:17: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 10-18 05:23:02 [__init__.py:244] Automatically detected platform cuda.


# 2 Labels Hate/Neutal- Lgb Models

## 2. Load LGB Training Data

Load `danghaidang-passau/lgb_Data`, which contains probability vectors from all four LLM annotators
for the human-labeled training samples. This dataset was produced by running `LLm_Labeling.ipynb`
on the 7 human-labeled training datasets. Each row has columns for each model's label probabilities
and the ground truth `label_id` from human annotation.

In [ ]:

repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_lgb = load_dataset(repo, "LightGBM_dataset")       
lgb_df = ds_lgb["train"]
lgb_df['ds'].value_counts()


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48598 [00:00<?, ? examples/s]

ds
HateSpeechX     15299
Sexism          10904
GermEval2019     9698
ViHSD            8061
GermEval2021     2071
US_election      1283
Covid            1282
Name: count, dtype: int64

In [ ]:
mstral7b = "mistral"
gemma9b = "gemma"
qwen14b = "qwen"
llama8B = "llama"
columns = [qwen14b, llama8B, mstral7b, gemma9b]

In [5]:
lgb_df['label_id'].value_counts()

label_id
2    32410
1    16188
Name: count, dtype: int64

In [6]:
lgb_df = lgb_df.sample(frac=1, random_state=def_seed)

## 3. Train LightGBM for Hate Label (Label 1)

Train a binary LightGBM classifier to predict whether a text is Hate (1) vs. not Hate.
Features: the per-LLM Hate-token probability column for each of the 4 models (`_label_1` suffix).
Target: `label_id == 1` (True = Hate, False = Neutral).
Early stopping is applied via the validation set (10% split). The trained model is saved to
`lgb_model_label_1.pkl`.

In [7]:
c1 = [x + "_label_1" for x in columns]
X = lgb_df[c1]
y = lgb_df['label_id'] == 1
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)



model_label_1 = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
)

val_preds = model_label_1.predict(X_val, num_iteration=model_label_1.best_iteration)
print(roc_auc_score(y_val, val_preds))


0.8649272837448871


In [11]:
with open("lgb_model_label_1.pkl", "wb") as f:
    pickle.dump(model_label_1, f)

## 4. Train LightGBM for Neutral Label (Label 2)

Train a second binary LightGBM classifier to predict whether a text is Neutral (2) vs. not Neutral.
Features: the per-LLM Neutral-token probability column for each model (`_label_2` suffix).
Target: `label_id == 2`. This model is used alongside `lgb_model_label_1` in `Ensemble_Labeling.ipynb`:
a text is labeled Hate if its Hate-score exceeds its Neutral-score and is positive.

In [8]:
c2 = [x + "_label_2" for x in columns]
X = lgb_df[c2]
y = lgb_df['label_id'] == 2
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)


model_label_2 = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
)

val_preds = model_label_2.predict(X_val, num_iteration=model_label_2.best_iteration)
print(roc_auc_score(y_val, val_preds))

0.8645123844871989


In [12]:
with open("lgb_model_label_2.pkl", "wb") as f:
    pickle.dump(model_label_2, f)